# **4_10_model_tft**

TFT: Temporal Fusion Transformer

## **Introducción y Resumen**

Temporal Fusion Transformer (TFT) fue propuesto por Google (Lim et al., 2020).
Está diseñado específicamente para series temporales multivariadas y combina lo mejor de varios mundos:

- LSTM → para capturar dependencias temporales locales.

- Self-Attention (Transformer) → para capturar relaciones de largo plazo y entre features.

- Gating + Variable Selection Networks → para seleccionar dinámicamente qué features son más relevantes en cada instante.

- Interpretabilidad → puedes visualizar la importancia temporal y por variable.

👉 En tu caso:

- Tienes ventanas fijas de 60 minutos (window_size=60).
- Cada ventana tiene muchas features (entre 900 y 1080), con relaciones complejas.
- Necesitas capturar patrones secuenciales y relevancia entre indicadores técnicos y alpha factors.

➡️ El TFT es ideal.


## **0. Configuración del Entorno**


### 0.1. Instalación de librerías


In [1]:
# ==============================================
# 1) ELIMINAR TODO LO VIEJO
# ==============================================
!pip uninstall -q -y torch torchvision torchaudio xformers lightning pytorch-forecasting pytorch-lightning

# ==============================================
# 2) INSTALAR PYTORCH GPU (CUDA 12.1) + TORCHVISION + TORCHAUDIO
# Compatible con Colab + Python 3.12
# ==============================================
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 \
torch==2.5.1

# ==============================================
# 3) INSTALAR LIGHTNING MODERNO + PYTORCH FORECASTING MODERNO
# (Compatibles con Python 3.12 y con el nuevo Lightning)
# ==============================================
!pip install -q "lightning>=2.2.0" "pytorch-forecasting"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 174.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 281.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 361.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 222.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 56.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 183.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 252.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 258.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 203.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 222.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 266.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 261.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

### 0.2. Importación de librerías


In [2]:
# ==============================
# Librerías de modelado (Lightning moderno)
# ==============================
import lightning.pytorch as pl
import torch

# ==============================
# PyTorch Forecasting (TFT y utilidades)
# ==============================
from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer,
)
from pytorch_forecasting.metrics import RMSE
from torch.utils.data import DataLoader

# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning tradicional
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

import joblib

warnings.filterwarnings("ignore")

In [3]:
import sys, platform
import numpy
import scipy
import sklearn
import torch
import lightning.pytorch as pl
import pytorch_forecasting

print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("Torch:", torch.__version__)
print("Lightning:", pl.__version__)
print("Forecasting:", pytorch_forecasting.__version__)

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.3
sklearn: 1.6.1
Torch: 2.5.1+cu121
Lightning: 2.6.0
Forecasting: 1.5.0


### 0.3. Acceso a Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.4. Comprobación de uso de RAM

In [5]:
import psutil

def ram_usage():
    ram = psutil.virtual_memory()
    used = ram.used / (1024**3)
    free = ram.available / (1024**3)
    total = ram.total / (1024**3)

    print(f"RAM total:      {total:.2f} GB")
    print(f"RAM usada:      {used:.2f} GB")
    print(f"RAM disponible: {free:.2f} GB")

In [6]:
ram_usage()

RAM total:      12.67 GB
RAM usada:      1.50 GB
RAM disponible: 10.86 GB


## **1. Carga de datos**

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [7]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [8]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [9]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [10]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [11]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
#features_to_30 = features_dict["features_to_30"]
#features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]


In [12]:
#print(f'Listado de features para 30min ({len(features_to_30)}): {features_to_30}')
#print(f'Listado de features para 60min ({len(features_to_60)}): {features_to_60}')
print(f'Listado de features para 90min ({len(features_to_90)}): {features_to_90}')

Listado de features para 90min (7): ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## **2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`**

### 2.0. Funciones

#### Función para cargar ventanas

In [13]:
def load_windows_and_scaler(target: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz'
    path_valid  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz'
    path_test   = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/3_dataset_preparation/global_scaler_{target[0]+target[1]}.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### Función para revisar información de ventanas

In [14]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

    return  X_train.shape[0], X_valid.shape[0], X_test.shape[0]

In [15]:
features_base = ['open', 'high', 'low', 'close', 'volume']

features_90 = features_base + features_to_90

window_size = 90

### 2.1 Carga de ventanas 90 minutos

In [16]:
ram_usage()

RAM total:      12.67 GB
RAM usada:      1.53 GB
RAM disponible: 10.83 GB


In [17]:
#X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90')
X_train, y_train, X_valid, y_valid, X_test, y_test, scaler = load_windows_and_scaler(target = '90')

In [18]:
n_samples_train_90, n_samples_valid_90, n_samples_test_90 = xy_info('90',  X_train, y_train, X_valid, y_valid, X_test, y_test)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	193487 targets.
	Distribución y: mean=0.000159, std=0.004746, min=-0.037742, max=0.039284

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000224, std=0.005887, min=-0.040748, max=0.083184

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=-0.000132, std=0.004558, min=-0.023392, max=0.026213


In [19]:
ram_usage()

RAM total:      12.67 GB
RAM usada:      3.74 GB
RAM disponible: 8.61 GB


## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [29]:
def load_metrics(data: str):
    data_path = f'{drive_path}/4_model_training/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [30]:
def metrics_verify(data: str) -> bool:
    data_path = f"{drive_path}/4_model_training/{data}.parquet"
    return os.path.exists(data_path)


In [31]:
def load_or_create_metrics (data:str):
  if metrics_verify(data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[4:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [32]:
tft_metrics, metrics = load_or_create_metrics("4_10_tft_metrics")

Las métricas no existen. Se crea el dataset _tft_metrics para almacenar las métricas


In [33]:
tft_metrics

,RMSE,MAE,R2,SMAPE,DirAcc


### 3.2. Función para guardar métricas

In [34]:
def save_metrics (metrics,  metrics_name: str):
  metrics_path = f"{drive_path}/4_model_training/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [35]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [36]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

In [37]:
tft_metrics

,RMSE,MAE,R2,SMAPE,DirAcc


## **4. Re-formateo más Encoder mínimo**

### **4.1. Helper: de 2D (aplanado) a 3D (B, T, F)**

Este bloque define una función auxiliar utilizada para **re-formatear las ventanas de datos** desde una representación 2D a la representación 3D requerida por el modelo.

- **Entrada**:
  - `X_flat`: matriz 2D con forma *(N, window_size × n_features)*.
  - `window_size`: longitud temporal de la ventana.
  - `n_features`: cantidad de features por paso temporal.

- **Funcionamiento**:
  - Verifica que la entrada sea efectivamente una matriz 2D.
  - Valida la consistencia dimensional comprobando que  
    `window_size × n_features == X_flat.shape[1]`.
  - Reconvierte los datos al formato *(N, window_size, n_features)* mediante `reshape`.

- **Salida**:
  - Un arreglo 3D listo para ser utilizado como entrada del modelo.

Este helper se utiliza para **cada conjunto de datos (train, valid y test)**.  
En este proyecto se emplea `window_size = 90` y `n_features_90 = 12`, asegurando que cada ventana temporal esté correctamente estructurada antes del entrenamiento.

In [38]:
import numpy as np
import torch
import torch.nn as nn
import math

def reshape_windows(X_flat: np.ndarray, window_size: int, n_features: int) -> np.ndarray:
    """
    Convierte X de (N, window_size * n_features) a (N, window_size, n_features).
    Valida la consistencia del producto.
    """
    assert X_flat.ndim == 2, "Se esperaba X_flat con 2D (N, T*F)."
    N, TF = X_flat.shape
    assert window_size * n_features == TF, (
        f"Inconsistencia: {window_size} * {n_features} != {TF}"
    )
    return X_flat.reshape(N, window_size, n_features)

En este bloque se definen los parámetros estructurales de entrada del modelo:

- `window_size = 90`  
  Establece la longitud de la ventana temporal, es decir, la cantidad de minutos consecutivos utilizados como entrada para cada muestra.

- `features_base`  
  Contiene las variables OHLCV básicas del mercado: *open, high, close, low y volume*.

- `features_90`  
  Se construye combinando las variables base con los factores adicionales definidos en `features_to_90`, conformando el conjunto completo de features utilizadas por el modelo.

- `n_features_90`  
  Representa la cantidad total de features por paso temporal y se obtiene como la longitud de `features_90`.

Este bloque permite **verificar explícitamente** el conjunto de features y su cardinalidad, asegurando coherencia dimensional con la configuración del modelo y las funciones de re-formateo de ventanas.

In [39]:
window_size = 90
features_base = ['open','high','close','low','volume']
features_90 = features_base + features_to_90
n_features_90 = len (features_90)
print(f'features_90:\t\t {features_90}')
print(f'n_features_90:\t {n_features_90}')

features_90:		 ['open', 'high', 'close', 'low', 'volume', 'ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']
n_features_90:	 12


**Re-formateo de ventanas y liberación de memoria**

Este bloque se encarga de **transformar las ventanas de entrada desde formato 2D a formato 3D**, de acuerdo con la configuración definida previamente (`window_size = 90` y `n_features_90 = 12`), y de **optimizar el uso de memoria** durante el procesamiento por fold.

- Para cada *fold* en `k_folds`:
  - Las matrices escaladas `X_train_sc[k]`, `X_valid_sc[k]` y `X_test_sc[k]`, originalmente en formato  
    *(N, window_size × n_features_90)*, se convierten al formato requerido por el modelo:  
    *(N, window_size, n_features_90)* mediante la función `reshape_windows`.
  - Los datos re-formateados se almacenan en los diccionarios `Xtr`, `Xva` y `Xte`, indexados por fold.

- Una vez completado el re-formateo de cada fold:
  - Se eliminan explícitamente las matrices 2D originales para evitar duplicación innecesaria de datos en memoria.
  - Se invoca el recolector de basura (`gc.collect()`) para liberar RAM de forma inmediata.

El objetivo principal es asegurar que cada conjunto de datos esté correctamente estructurado para el entrenamiento del modelo Transformer, manteniendo un consumo de memoria controlado durante el procesamiento de múltiples folds.

In [40]:
ram_usage()

RAM total:      12.67 GB
RAM usada:      3.77 GB
RAM disponible: 8.58 GB


In [41]:
Xtr = reshape_windows(X_train, window_size, n_features_90)
Xva = reshape_windows(X_valid, window_size, n_features_90)
Xte = reshape_windows(X_test, window_size, n_features_90)

In [42]:
ytr = y_train.copy()
yva = y_valid.copy()
y_te = y_test.copy()

In [43]:
import gc
del X_train, X_valid, X_test, y_train, y_valid, y_test
gc.collect()

144

**Verificación de dimensiones de entrada por fold**

Este bloque define una función auxiliar destinada a **verificar las dimensiones de los datos de entrada** del modelo para cada fold.

- La función itera sobre los *folds* definidos en `k_folds`.
- Para cada fold:
  - Muestra de forma ordenada los *shapes* de los conjuntos **train**, **valid** y **test**.
  - Verifica que cada conjunto se encuentre en formato 3D, consistente con la estructura  
    *(n_samples, window_size, n_features)* requerida por el modelo.

- La salida se presenta en forma tabular, facilitando la inspección visual y la detección temprana de inconsistencias dimensionales entre folds o conjuntos de datos.

El objetivo principal es confirmar que el re-formateo de las ventanas se haya realizado correctamente antes de proceder al entrenamiento y evaluación del modelo.


In [44]:
MOSTRAR_SHAPES_FOLDS = '''
def mostrar_shapes_folds_simple(k_folds, Xtr, Xva, Xte):
    for k in k_folds:
        print(f"\nShapes del Fold {k}")
        print(f"{'Set':<10}{'Shape (3D)':<25}")
        print("-" * 40)

        filas = [
            ("Train", Xtr[k].shape),
            ("Valid", Xva[k].shape),
            ("Test",  Xte[k].shape),
        ]

        for nombre, shape_3d in filas:
            print(f"{nombre:<10}{str(shape_3d):<25}")
            '''

In [45]:
def mostrar_shapes_simple(Xtr, Xva, Xte):
    #for k in k_folds:
        print(f"\nShapes 3D de ventanas:") #del Fold {k}")
        print(f"{'Set':<10}{'Shape (3D)':<25}")
        print("-" * 40)

        filas = [
            ("Train", Xtr.shape),
            ("Valid", Xva.shape),
            ("Test",  Xte.shape),
        ]

        for nombre, shape_3d in filas:
            print(f"{nombre:<10}{str(shape_3d):<25}")

In [46]:
mostrar_shapes_simple(Xtr, Xva, Xte)


Shapes 3D de ventanas:
Set       Shape (3D)               
----------------------------------------
Train     (193487, 90, 12)         
Valid     (41567, 90, 12)          
Test      (41567, 90, 12)          


## **5. Conversión de ventanas temporales 3D a DataFrame compatible con TFT**

### **5.1. Descripción funcional**

Es siguiente código convierte un dataset de ventanas temporales en formato 3D en un DataFrame en formato long compatible con Temporal Fusion Transformer (TFT) de pytorch-forecasting.

El objetivo es adaptar los datos al esquema requerido por la librería, donde cada fila representa un paso temporal dentro de una serie.

---

**Estructura de entrada**

El código parte de los siguientes arreglos:

- X con forma (N, T, F)
  - N: número de ventanas o series temporales
  - T: cantidad de pasos temporales por ventana
  - F: número de features por paso temporal
- y con forma (N,)
  - Un valor objetivo por cada ventana

---

**Estructura de salida**

Se construye un DataFrame donde:

- Cada fila corresponde a un paso temporal
- Cada ventana se identifica mediante un group_id
- El índice temporal se representa con time_idx
- El target se repite en todos los pasos de la ventana
- Las features se aplanan en columnas feat_0, feat_1, …, feat_(F-1)

Este formato es exactamente el esperado por pytorch-forecasting para entrenar modelos como Temporal Fusion Transformer.

### **5.2. Código: construcción del DataFrame para TFT**

In [47]:
import pandas as pd
import numpy as np

def build_tft_df(X, y, prefix="feat"):
    """
    Convierte ventanas temporales 3D en un DataFrame long
    compatible con pytorch-forecasting (TFT).

    Parámetros
    ----------
    X : np.ndarray
        Array de forma (N, T, F)
        N = número de ventanas / series
        T = longitud temporal
        F = número de features por paso temporal

    y : np.ndarray
        Array de forma (N,)
        Target por cada ventana

    prefix : str
        Prefijo para las columnas de features (ej: feat_0, feat_1, ...)
    """

    # Extraemos dimensiones
    N, T, F = X.shape

    # Columnas base del DataFrame
    data = {
        # time_idx: 0..T-1 repetido N veces (una por cada ventana)
        "time_idx": np.tile(np.arange(T), N),

        # group_id: id de la ventana (0..N-1), cada uno repetido T veces
        "group_id": np.repeat(np.arange(N), T),

        # target: valor objetivo por ventana, repetido en todos los pasos temporales
        "target": np.repeat(y, T),
    }

    # Aplanamos las features:
    # X[:, :, j] tiene forma (N, T)
    # reshape(-1) → (N*T,)
    for j in range(F):
        data[f"{prefix}_{j}"] = X[:, :, j].reshape(-1)

    # Construimos el DataFrame final
    df = pd.DataFrame(data)

    return df


### **5.3. Generación de DataFrames de entrenamiento y validación para TFT**

A partir de las ventanas temporales 3D y sus targets asociados, se construyen los DataFrames en formato long requeridos por pytorch-forecasting para entrenar un Temporal Fusion Transformer (TFT).

En este paso se generan los conjuntos de entrenamiento y validación, manteniendo el mismo prefijo de features para asegurar consistencia entre ambos datasets.

In [48]:
df_tr = build_tft_df(Xtr, ytr, prefix="feat")
df_va = build_tft_df(Xva, yva, prefix="feat")

Ambos DataFrames comparten la misma estructura (time_idx, group_id, target y features f90_*), condición necesaria para el correcto entrenamiento del modelo TFT.

## **6. Definición de variables para `TimeSeriesDataSet`**

En este paso se definen las listas de variables requeridas por TimeSeriesDataSet de pytorch-forecasting, separando las features temporales y el target según su naturaleza.

Para el horizonte de 90 minutos, todas las features se consideran variables reales conocidas en el tiempo, mientras que el target se define como variable real desconocida.

In [49]:
feature_cols = [c for c in df_tr.columns if c.startswith("feat_")]

time_varying_known_reals   = feature_cols
time_varying_unknown_reals = ["target"]

static_reals        = []
static_categoricals = []

- `time_varying_known_reals_90`: features disponibles en todos los pasos temporales
- `time_varying_unknown_reals_90`: variable objetivo a predecir
- `static_reals_90 / static_categoricals_90`: no se utilizan variables estáticas en este setup

Estas listas se utilizarán directamente para instanciar el objeto TimeSeriesDataSet del modelo TFT.

## **7. Creación del `TimeSeriesDataSet` para TFT (horizonte 90 minutos)**

En este paso se instancia el objeto TimeSeriesDataSet de pytorch-forecasting, que define cómo el Temporal Fusion Transformer (TFT) consume las series temporales durante el entrenamiento.

Se utiliza un historial de 90 minutos como encoder y se define un horizonte de predicción de 1 paso, correspondiente al retorno a 90 minutos.

### **7.1. Configuración de longitudes temporal**

Nuestras ventanas tienen 90 pasos temporales, y en pytorch-forecasting:
  - `max_encoder_length` = cantidad de pasos usados como entrada
  - `max_prediction_length` = cantidad de pasos a predecir

In [50]:
max_encoder_length = 89      # historial temporal (90 minutos)
max_prediction_length = 1   # retorno a 90 min como un único valor

el modelo ve:
- 89 pasos como historial (encoder)
- 1 paso como predicción (decoder)

- Total efectivo: 90 pasos

Esto es coherente si:
- El target representa el retorno a 90 minutos
- Y ese retorno se asocia al último paso temporal de la ventana

#### **Algunas aclaraciones:**

El modelo utiliza 89 pasos temporales como historial (encoder) y 1 solo paso para la predicción (decoder). Dado que el target está repetido en todos los time_idx dentro de cada ventana, el Temporal Fusion Transformer (TFT) aprende a:

**Predecir el valor del target asociado al último paso temporal de la ventana, es decir, el valor correspondiente a la última fila de cada serie.**

**Cómo construye la muestra el `TimeSeriesDataSet`**

- El encoder utiliza los pasos `time_idx = 0 … 88`
- El decoder contiene un único paso: `time_idx = 89`
- El valor del target utilizado para el entrenamiento corresponde a ese paso del decoder

Como el `target` es constante dentro de la ventana, esto es equivalente a entrenar con el target de la última fila.

**Relación con el problema planteado**

Cada ventana representa:
- 90 minutos de información histórica
- Un único valor objetivo (retorno a 90 minutos)

El modelo aprende la siguiente relación: “Dado el historial completo de la ventana, predecir el retorno futuro asociado a esa ventana”.

**Nota técnica:**

Repetir el target en todos los pasos temporales:
- No afecta negativamente el entrenamiento
- Aunque no es estrictamente necesario

En formulaciones más canónicas, el `target` podría definirse solo en el último `time_idx`.

Para este caso, la implementación utilizada es correcta, consistente y funcional.

### **7.2. Creación del TimeSeriesDataSet**

Para entrenamiento:

In [51]:
training_90 = TimeSeriesDataSet(
    df_tr,                     # DataFrame en formato long con las series de entrenamiento
    time_idx="time_idx",           # Columna que indica el índice temporal dentro de cada serie
    target="target",               # Variable objetivo a predecir
    group_ids=["group_id"],        # Identificador de cada serie temporal (una por ventana)

    max_encoder_length=max_encoder_length,        # Longitud máxima del encoder (historial: 90 minutos)
    max_prediction_length=max_prediction_length,  # Horizonte de predicción (1 paso: retorno a 90 min)

    # Variables reales conocidas en el tiempo (features disponibles en cada paso)
    time_varying_known_reals=time_varying_known_reals,

    # Variables reales desconocidas en el tiempo (incluye el target)
    time_varying_unknown_reals=time_varying_unknown_reals,

    # No se utilizan variables estáticas reales en este setup
    static_reals=static_reals,

    # No se utilizan variables categóricas estáticas en este setup
    static_categoricals=static_categoricals,

    # Se desactiva la normalización automática del target
    # (el escalado se controla externamente)
    target_normalizer=None,
)

Este dataset define:
- La estructura temporal de las series (time_idx, group_id)
- El tamaño del encoder y del horizonte de predicción
- La separación entre variables conocidas, desconocidas y estáticas
- La ausencia de normalización automática del target (control externo)

El objeto resultante será utilizado directamente para crear los DataLoader y entrenar el modelo TFT.

Para validación:

In [53]:
validation_90 = TimeSeriesDataSet.from_dataset(
    training_90,              # Dataset base: reutiliza la misma configuración del training
    df_va,                 # DataFrame en formato long para validación
    predict=False,            # Indica que es un dataset de validación (no de inferencia)
    stop_randomization=True,  # Desactiva la aleatorización para obtener validación determinística
)

Notas clave:

- ` from_dataset` garantiza que train y validación tengan exactamente la misma estructura.
- `stop_randomization=True` es fundamental para métricas estables y reproducibles.
- `predict=False` asegura que el target esté disponible para evaluación.

Este bloque es el patrón correcto y recomendado para validación en TFT.

## **8.Creación de `DataLoaders` para entrenamiento y validación**

En este paso se generan los `DataLoaders` a partir de los objetos `TimeSeriesDataSet`, que serán utilizados por el modelo TFT durante el entrenamiento y la validación.

Se define un `batch_size` y se configura:
- Entrenamiento con `shuffle=True` para mezclar las series y mejorar la generalización.
- Validación con `shuffle=False` para mantener una evaluación determinística y reproducible.

In [ ]:
ram_usage()

In [54]:
batch_size = 256

train_dataloader_90 = training_90.to_dataloader(
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

val_dataloader_90 = validation_90.to_dataloader(
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

In [ ]:
ram_usage()

## **9. Definición del modelo Temporal Fusion Transformer (TFT)**

En este paso se configura el modelo Temporal Fusion Transformer a partir del TimeSeriesDataSet de entrenamiento (training_90).
Además, se desactiva cuDNN para evitar posibles inconsistencias o errores con ciertas operaciones en GPU durante el entrenamiento.

In [55]:
import torch

# Desactivar cuDNN (opcional): útil si se presentan errores o comportamientos no deterministas
torch.backends.cudnn.enabled = False
print("cuDNN enabled:", torch.backends.cudnn.enabled)

# Definición del modelo TFT a partir del dataset
tft_90 = TemporalFusionTransformer.from_dataset(
    training_90,
    hidden_size=16,                           # tamaño del estado oculto (capacidad del modelo)
    attention_head_size=2,              # número de cabezas de atención
    hidden_continuous_size=8,       # proyección interna para variables continuas
    lstm_layers=1,                              # cantidad de capas LSTM internas

    dropout=0.1,                                 # dropout para regularización
    learning_rate=1e-3,                     # tasa de aprendizaje
    loss=RMSE(),                                 # función de pérdida (RMSE)
    reduce_on_plateau_patience=3,   # paciencia para reducir LR si no mejora

    log_interval=10,                            # frecuencia de logging durante entrenamiento
    log_val_interval=1,                       # logging de validación cada 1 época
)

cuDNN enabled: False


## **10. Entrenamiento del modelo TFT**

En este punto se entrena el modelo Temporal Fusion Transformer (TFT) utilizando PyTorch Lightning. Se plantean dos etapas:

Entrenamiento 1 (corto): corrida rápida para verificar que el pipeline funciona correctamente (datos, modelo, GPU, métricas).

Entrenamiento 2 (largo): entrenamiento completo incorporando Early Stopping y monitoreo del Learning Rate, para detener el entrenamiento cuando la validación deje de mejorar y registrar la evolución del LR.

### **10.1. Entrenamiento 1 (corrida rápida de verificación)**

In [56]:
#Entrenamiento 1

trainer_90_0 = pl.Trainer(
    max_epochs=5,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    precision=32,             # Mantener 32 bits para estabilidad
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    log_every_n_steps=10,
)

trainer_90_0.fit(
    tft_90,
    train_dataloaders=train_dataloader_90,
    val_dataloaders=val_dataloader_90,
)

INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ RMSE                            │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │      0 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    208 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │      0 │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  9.0 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  8.2 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    808 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     17 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 31.0 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 31.0 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 506                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 465, in advance
    self.epoch_loop.run(self._data_fetcher)
  File

TypeError: object of type 'NoneType' has no len()

### **10.2. Early Stopping y monitoreo del Learning Rate (para Entrenamiento 2)**

In [ ]:
# Early Stopping para segundo entrenamiento

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
import torch

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min"
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

### **10.3. Entrenamiento 2 (entrenamiento completo con Early Stopping)**

In [ ]:
# Entrenamiento 2

trainer_90_1 = pl.Trainer(
    max_epochs=30,              # en vez de 5
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    precision=32,
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    log_every_n_steps=10,
    callbacks=[early_stop, lr_monitor],
)

trainer_90_1.fit(
    tft_90,
    train_dataloaders=train_dataloader_90,
    val_dataloaders=val_dataloader_90,
)

print("Epoch actual:", trainer_90_1.current_epoch)
print("Épocas completadas:", trainer_90_1.fit_loop.epoch_progress.current.completed)


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | RMSE                            | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 0      | train
3  | prescalers                         | ModuleDict                      | 208    | train
4  | static_variable_selection          | VariableSe

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch actual: 1
Épocas completadas: 1


## **11. Predicciones con el modelo TFT**


En este punto se definen las funciones necesarias para obtener las predicciones del modelo TFT sobre el conjunto de validación.

El objetivo es extraer, de forma controlada y sin gradientes, los valores predichos por el modelo y los valores reales del target, para luego poder evaluar métricas.

### **11.1. Obtener predicciones**

La siguiente función:

- Pone el modelo en modo evaluación (eval)
- Itera sobre el DataLoader de validación
- Extrae las predicciones del output del TFT
- Maneja correctamente los distintos formatos de salida de pytorch-forecasting
- Devuelve y_true y y_pred como arrays de NumPy, listos para evaluación

In [ ]:
def eval_tft(tft_n, val_dataloader_n):

    # Modo evaluación (desactiva dropout, etc.)
    tft_n.eval()

    y_true_list = []
    y_pred_list = []

    # Iterar sobre el DataLoader de validación
    for batch_x, batch_y in iter(val_dataloader_n):

        # Desactivar cálculo de gradientes
        with torch.no_grad():
            out = tft_n(batch_x)

        # Extraer el tensor de predicción del output
        if hasattr(out, "prediction"):
            preds = out.prediction            # caso típico pytorch_forecasting
        elif isinstance(out, dict) and "prediction" in out:
            preds = out["prediction"]         # por si viene como dict
        else:
            preds = out                       # fallback: ya es un Tensor

        # batch_y puede venir como (y, weight) o solo y
        if isinstance(batch_y, tuple):
            y = batch_y[0]
        else:
            y = batch_y

        # Acumular resultados
        y_true_list.append(y)
        y_pred_list.append(preds)

    # Concatenar batches y convertir a NumPy
    y_true = torch.cat(y_true_list, dim=0).detach().cpu().numpy().ravel()
    y_pred = torch.cat(y_pred_list, dim=0).detach().cpu().numpy().ravel()

    return y_true, y_pred


Esta función permite evaluar el modelo TFT de forma consistente y reutilizable, manteniendo el control total sobre la inferencia y el postprocesamiento de las predicciones.

### **11.2. Evaluación del modelo TFT**

En este paso se obtienen las predicciones del modelo TFT sobre el conjunto de validación y luego se calculan las métricas de desempeño utilizando la función `evaluate_model`.

In [ ]:
# Obtener predicciones sobre validación
y_true_90, y_pred_90 = eval_tft(tft_90, val_dataloader_90)

# Calcular métricas (RMSE, MAE, R2, SMAPE, DirAcc, etc.)
tft_90_metrics = evaluate_model(
    model=None,
    X=None,
    y_true=y_true_90,
    y_pred=y_pred_90
)

print(tft_90_metrics)

{'RMSE': 0.004776321351528168, 'MAE': 0.003908372949808836, 'R2': -0.2322402000427246, 'SMAPE': 129.94976806640625, 'DirAcc': 0.6622137404580153}


- `y_true_90`: valores reales del target en validación
- `y_pred_90`: predicciones generadas por el TFT
- `tft_90_metrics`: diccionario/tabla con las métricas calculadas para el modelo TFT

In [ ]:
tft_metrics.loc['TFT'] = tft_metrics

In [ ]:
tft_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
TFT_30_subsampleado_10%_1,0.002582,0.001922,0.023397,129.001556,0.631202
TFT_30_subsampleado_10%_2,0.002985,0.002532,-0.304982,131.029831,0.625477
TFT_60_subsampleado_10%_1,0.003997,0.003385,-0.237267,132.961823,0.654819
TFT_60_subsampleado_10%_2,0.005539,0.004507,-1.376273,149.686584,0.444418
TFT_90_subsampleado_10%_1,0.002139,0.002110,0.752847,113.214897,0.712309
TFT_90_subsampleado_10%_2,0.004776,0.003908,-0.232240,129.949768,0.662214


In [ ]:
save_metrics (tft_metrics, "4_10_tft_metrics")

Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_10_tft_metrics.parquet
